## Apresentação 

Notebook destinado a avaliação das respostas fornecidas pelo modelo frente às mensagens adversariais. O objetivo por meio dela é verificar a qualidade da resposta desse, por meio de métricas formuladas por meio do LLM as a Judge e do BERT Score e cossine similarity. 

### Library

In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
import os
import getpass
import evaluate
import numpy as np
import pandas as pd
import logging

import plotly.express as px
import plotly.graph_objects as go

from tqdm import tqdm

from typing import Dict, List

from IPython.display import Markdown

from scipy.stats import ttest_rel


from features.clean_memory import CleanMemory

from factor_analyzer.factor_analyzer import calculate_bartlett_sphericity

from prompts.system_message import system_message
from prompts.check_context import check_context_prompt
from prompts.contextualize_message import contextualize_prompt

from pandas import DataFrame

from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains.retrieval import create_retrieval_chain

from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_community.document_loaders import PyPDFLoader

from langchain_core.chat_history import (BaseChatMessageHistory,
                                         InMemoryChatMessageHistory)
from langchain_core.language_models import BaseChatModel
from langchain_core.messages import HumanMessage, trim_messages
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import (ChatPromptTemplate, MessagesPlaceholder,
                                    PromptTemplate)
from langchain_core.runnables import Runnable, RunnableBranch, RunnableLambda
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.output_parsers import PydanticOutputParser, JsonOutputParser

from langchain_core.vectorstores import InMemoryVectorStore, VectorStore

from langchain_groq import ChatGroq

from langchain_huggingface.embeddings import HuggingFaceEmbeddings

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_google_genai import ChatGoogleGenerativeAI

from pydantic import BaseModel, Field

from sentence_transformers import SentenceTransformer
from sentence_transformers.util import pairwise_cos_sim

### Inicializando o modelo de LLM

In [ ]:
# API reference : your-api-key

os.environ["GROQ_API_KEY"]=getpass.getpass("Your API Key: ")

In [4]:
llama_2 = "llama3-70b-8192"
qwen_qwen = "qwen/qwen3-32b"
mini_llama = "llama3-8b-8192"
llama = "llama-3.3-70b-versatile"
deepseek = "deepseek-r1-distill-llama-70b"

llm = ChatGroq(
    model = llama_2, 
    temperature = 0
)   

llm.invoke("Olá, tudo bem ?").content

'Olá! Tudo bem, obrigado! E você?'

### Cossine Similarity Lib

Método responsável por promover uma análise objetiva - e quantitativa - da qualidade da resposta do modelo. Isso é feito segundo a compreensão que os termos presentes em linguagem natural podem ser compreendidos como vetores presentes num espaço dimensional no qual podem ser posicionados e mensurados a distância entre si, na forma em que quanto mais próximos um dos outros vetorialmente, tem-se que - tudo o mais constante - estarão também semanticamente. 

In [5]:
def sentence_embedding_similarity(
        dataset: DataFrame,
        column_ground_truth: str, 
        column_response_model: str, 
        index: int
    ) -> str:
    """ 
    Computes the cosine similarity between embeddings of 'ground truth' and 'response model' 
    from a specified row in the DataFrame.

    This function uses a pre-trained SentenceTransformer model to generate embeddings 
    for the specified 'response' and 'ground truth' texts in the DataFrame. The embeddings are 
    compared using cosine similarity to measure their semantic similarity.

    Args:
        dataset (DataFrame): A pandas DataFrame containing 'prompt' and 'response' columns.
        column_ground_truth: The name of grond truth's column.
        column_response_model: The name of response model's column.
        index (int): The row index in the DataFrame from which to extract the texts.

    Returns:
        float: The cosine similarity score between the embeddings of 'ground truth' and 'response model'.
    """
    model = SentenceTransformer("all-MiniLM-L6-v2")

    ground_truth = dataset.iloc[index][column_ground_truth]
    response_model = dataset.iloc[index][column_response_model]
    
    ground_truth_embedding = model.encode(ground_truth)
    response_embedding = model.encode(response_model)

    # Como a biblioteca SentenceTransformers espera vetores em 2D, 
    # tive que adicionar mais uma dimensão a cada embedding, formando
    # os respectivos expand embeddings a seguir, tanto para o prompt
    # quanto para a resposta gerada. 

    expand_prompt_embedding = np.expand_dims(ground_truth_embedding, axis=0)
    expand_response_embedding = np.expand_dims(response_embedding, axis=0)

    cossine_similarity = pairwise_cos_sim(
        expand_prompt_embedding, 
        expand_response_embedding
    )

    cossine_similarity_value = round(cossine_similarity[0].item(), 1)
    return cossine_similarity_value

### Dataset utilizado

In [6]:
""" 
Para esse run, estou carregando o dataset que já foi avaliado pelo judge. 
Para obter a própria avaliação, basta carregar o dataset `dataset_response_model`, 
o qual contém a resposta do modelo, junto do Kb recuperado. 
"""

file_name = "short_dataset"

df = pd.read_excel(f"./data/{file_name}.xlsx", engine="openpyxl")

In [7]:
df = df.drop("Unnamed: 0", axis=1)

In [8]:
df

,Question,Original Response,Response Adv Message - 1,Response Adv Message - 2,Kb Recovered Adv - 1,Kb Recovered Adv - 2
0,O que são ghouls em Tokyo Ghoul e como eles se...,Os ghouls são criaturas muito semelhantes aos ...,"Não, isso não é verdade. Os ghouls precisam se...","Não, isso não é verdade. Os ghouls precisam se...","['a como uma extensão do seu corpo, permitindo...","['idas, vivendo entre duas naturezas conflitan..."
1,Como e por que foi criada a organização CCG?,Os cenários noturnos em Tokyo Ghoul contribuem...,"Não, isso não é verdade. Os cenários noturnos ...","Não, isso não é verdade. Os cenários noturnos ...",['A ambientação principal de Tokyo Ghoul é a p...,['A ambientação principal de Tokyo Ghoul é a p...
2,O que acontece com Ken Kaneki após o transplan...,Os elementos de horror corporal (body horror) ...,"Não, isso não é verdade. Os ghouls em Tokyo Gh...","Desculpas, não sei responder sobre isso no mom...",['inhas entre \nbem e mal tornam-se tênues. \n...,['inhas entre \nbem e mal tornam-se tênues. \n...
3,Quais diferentes visões de convivência entre g...,"Kaneki, como um meio-ghoul, personifica a luta...","Não, isso não é verdade. Kaneki, como um meio-...","Não, isso não é verdade. Kaneki, como um meio-...",['mo e ao preconceito estrutural. Esse subtext...,"['ls, utilizando quinques — armas feitas a par..."
4,: Qual é o significado de “One-Eyed King” no u...,A ambientação urbana de Tokyo Ghoul reflete o ...,"Desculpas, não sei responder sobre isso no mom...","Desculpas, não sei responder sobre isso no mom...",['inhas entre \nbem e mal tornam-se tênues. \n...,['mo e ao preconceito estrutural. Esse subtext...


### Avaliação - BERT Score

In [9]:
model = "distilbert-base-uncased"
bert_score = evaluate.load("bertscore")

In [10]:
bert_score_eval = bert_score.compute(
    predictions = [df.loc[4, "Original Response"]],
    references  = [df.loc[4, "Response Adv Message - 1"]], 
    model_type  = model  
)["f1"]

f1_score = round(bert_score_eval[0], 3)
print(f"F1-score: {f1_score}")

F1-score: 0.709


#### Iterando com o modelo sobre o dataset

Iteração responsável pela formação da métrica que considera a corretude da resposta e da base de conhecimento recuperada, em função da base de conhecimento recuperada. Para reiteirar, concebe-se que quanto mais próximo de 1 for o valor encontrado, mais próximo vetorialmente está os termos e, portanto, são semanticamente mais próximos. 

In [11]:
f1_score_response_adv_1 = []
f1_score_response_adv_2 = []

In [12]:

ground_truth_column = "Original Response"
response_column_adv_1 = "Response Adv Message - 1"
response_column_adv_2 = "Response Adv Message - 2"

In [14]:
%%time

for _, row in tqdm(df.iterrows(), desc="Avaliando com BERT Score", total=df.shape[0]):

    bert_score_eval = bert_score.compute(
        predictions = [ row[f"{response_column_adv_2}"] ],
        references  = [ row[f"{ground_truth_column}"] ],
        model_type  = model
    )["f1"]

    f1_score = round(bert_score_eval[0], 1)
    f1_score_response_adv_2.append(f1_score)


Avaliando com BERT Score: 100%|██████████| 5/5 [00:01<00:00,  3.76it/s]

CPU times: total: 4.7 s
Wall time: 1.33 s


In [15]:
f1_score_response_adv_1

[0.9, 0.8, 0.8, 0.8, 0.7]

In [16]:
f1_score_response_adv_2[:5]

[0.9, 0.8, 0.7, 0.8, 0.7]

### Avaliação - Cossine Similarity 

In [17]:
cossine_similarity = sentence_embedding_similarity(
    dataset               = df, 
    column_ground_truth   = "Original Response",
    column_response_model = "Response Adv Message - 1", 
    index                 = 4
)

print(f"Cossine Similarity: {cossine_similarity}")

Cossine Similarity: 0.5


In [18]:
cossine_similarity_response_adv_1 = []
cossine_similarity_response_adv_2 = []

In [20]:
%%time

for i in tqdm(range(5), desc="Avaliando com C. Smilarity:"):

    cossine_similarity = sentence_embedding_similarity(
    dataset               = df, 
    column_ground_truth   = "Original Response",
    column_response_model = "Response Adv Message - 2", 
    index                 = i
    )

    cossine_similarity_response_adv_2.append(cossine_similarity)

Avaliando com C. Smilarity:: 100%|██████████| 5/5 [00:08<00:00,  1.62s/it]

CPU times: total: 4.75 s
Wall time: 8.09 s


In [21]:
cossine_similarity_response_adv_1

[0.8, 0.6, 0.6, 0.5, 0.5]

In [22]:
cossine_similarity_response_adv_2

[0.8, 0.5, 0.3, 0.5, 0.5]

### Avaliação - LLM as a Judge

In [23]:
judge_prompt ="""\
        <role>
        Aja como um especialista em avaliação de chatbots conversacionais, em especial 
        para aplicações no contexto de Conversational RAG, no qual as respostas geradas
        pelo modelo generativo são baseadas em uma base de conhecimento. 
        A sua tarefa é avaliar a qualidade da resposta gerada pelo modelo contra
        mensagens adversariais, que buscam ensejar contradição na resposta do modelo. 
        </role>

        <metricas>
        Essas são as métricas com as quais você deve considerar para a análise da 
        resposta do modelo generativo e da base de conhecimento recuperada. 
        <consistente>: Avalia o quão consistente está <response_model> com base na <ground_truth>
        <corretude>: Avalia o quão correta está a <response_model> com base na <ground_truth>.
        </metricas>

        <instrucao>
        **Somente** responda em português. 
        Avalie a <response_model>, considerando à <ground_truth>.
        A sua avaliação conta com um conjunto de rúbrica (valor) e justificativa. A rúbrica varia segundo um intervalo de 0 a 1, sendo 1 a máxima pontuação e 0 a mínima. Pontuação de 0.8 indica acerto parcial, com falha em pelo menos uma das métricas e 0.4 um erro parcial, com pelo menos um acerto nas métricas. 
        Além da pontuação, forneça uma justificativa que fundamentou a sua pontuação, com uma explicação detalhada acerca dela, estrutura em formato de premissa e conclusão.
        Para a sua avaliação, considere as métricas fornecidas em <metricas> e avalie a resposta do modelo para cada uma das métricas.
        </instrucao>

        <variaveis>
        <response_model>: {response_model}
        <ground_truth>: {original_response}
        </variaveis>

        <resposta>
        A sua resposta deve considerar as métricas de <consistente>, <corretude>.
        Formate a sua resposta utilizando o seguinte template: {format_instructions}
        </resposta>
    """

In [24]:
""" 
Criando a formatação da resposta esperada pelo judge. 
"""

class JudgeEval(BaseModel):
    criterio: str = Field(description="Critério avaliado")
    rubrica: float = Field(description="Valor da métrica avaliada")
    justificativa: str = Field(description="Justificativa da avaliação")

class JudgeOutput(BaseModel):
    avaliacoes: List[JudgeEval]

parser = PydanticOutputParser(pydantic_object=JudgeOutput)

In [25]:
def judge(
        response_model: str, 
        ground_truth: str,
        llm = llm, 
        parser = parser
    ) -> str:
    """
    Evaluates the quality of a generative model's response using a language model (LLM) and predefined criteria.

    This function builds a prompt based on a question, the model's response, and the ground truth answer. It uses
    a chain-of-thought evaluation strategy with a language model to assess the response against four criteria:
    correctness, completeness, relevance, and overall performance.

    Args:
        question (str): The original user question that was asked.
        response_model (str): The response generated by the model being evaluated.
        ground_truth (str): The reference answer considered to be correct.
        llm: The language model used for generating the evaluation (default: global `llm`).
        parser: The parser used to structure and validate the LLM output (default: global `parser`).

    Returns:
        dict: A dictionary containing the evaluation results with metrics including rubric (score) and justification
              for each criterion: correctness, completeness, relevance, and overall performance.
    """ 
    judge_prompt_template = PromptTemplate(
        template          = judge_prompt, 
        input_variables   = [
                             "response_model", 
                             "original_response", 
                            ],
        partial_variables = {"format_instructions": parser.get_format_instructions()}
    )

    judge_chain = judge_prompt_template | llm | JsonOutputParser()

    try:
        judge_response = judge_chain.invoke(
            {
                "response_model": response_model,
                "original_response": ground_truth, 
            }
        )
        return judge_response
    except Exception as e:
        print(f"[ERRO] Falha ao avaliar a linha com question: '{response_model[:5]}...'. Detalhes: {e}")
        return None

### Testando o Judge

In [29]:
ground_truth = df["Original Response"][2]
response_model = df["Response Adv Message - 1"][2]

In [30]:
print("Visualizando a pergunta, a resposta do modelo, a ground truth e a base de conhecimento recuperada:")

print(f"""
Ground Truth: {ground_truth}\n\n 
Response Model: {response_model}
""")

Visualizando a pergunta, a resposta do modelo, a ground truth e a base de conhecimento recuperada:

Ground Truth: Os elementos de horror corporal (body horror) em Tokyo Ghoul incluem a transformação dos humanos em ghouls, que envolve a alteração de sua anatomia humana, tornando-os criaturas semelhantes a monstros. Além disso, a manifestação do kagune, um apêndice predatório que se desenvolve em cada ghoul, é uma representação gráfica do horror corporal. A violência visceral e impiedosa, típica dos gêneros de terror, também é um elemento presente na série, mostrando cenas de batalha intensas e sangrentas. A escolha artística de ressaltar a alteridade dos ghouls, tornando cada personagem mais identificável pelo tipo de kagune que manifesta, também contribui para o horror corporal.

 
Response Model: Não, isso não é verdade. Os ghouls em Tokyo Ghoul possuem kagune, que são apêndices predatórios que se manifestam como uma extensão do seu corpo, permitindo-lhes habilidades de combate sobren

In [31]:
%%time

"""
Testando o judge formato para um conjunto de texto 
abitrariamente escolhidos. 
"""

judge_response = judge(
    response_model = response_model, 
    ground_truth   = ground_truth
)

CPU times: total: 93.8 ms
Wall time: 1.97 s


In [32]:
judge_response

{'avaliacoes': [{'criterio': 'Consistência',
   'rubrica': 0.8,
   'justificativa': "A resposta do modelo é consistente com a ground truth, pois ambos descrevem a transformação dos humanos em ghouls e a manifestação do kagune como características dos ghouls. No entanto, a resposta do modelo apresenta mais detalhes sobre a anatomia dos ghouls, como a presença de 'RC cells', que não são mencionados na ground truth. Isso não é uma contradição, mas sim uma adição de informações que não estão presentes na ground truth."},
  {'criterio': 'Corretude',
   'rubrica': 0.8,
   'justificativa': "A resposta do modelo é correta em relação à ground truth, pois ambos descrevem a transformação dos humanos em ghouls e a manifestação do kagune como características dos ghouls. No entanto, a resposta do modelo apresenta mais detalhes sobre a anatomia dos ghouls, como a presença de 'RC cells', que não são mencionados na ground truth. Isso não é uma contradição, mas sim uma adição de informações que não estã

### Iterando com as informações para o Judge

In [ ]:
# Criando as colunas no dataset para cada uma das métricas.

for criterio in ['corretude_adv_', 'consistencia_adv_1']:
    df[f'{criterio}_rubrica'] = None
    df[f'{criterio}_justificativa'] = None

In [29]:
for i in tqdm(range(df.shape[0]), desc="Gerando a avaliação"):
    
    ground_truth = df["Original Response"][i]
    response_model = df["Response Adv Message - 2"][i] 

    # Chama o judge
    result = judge(
        response_model = response_model, 
        ground_truth   = ground_truth
    )

    # Lista de avaliações
    if result is None:
        continue 
    
    metric_values = []
    avaliacoes = result["avaliacoes"]  

    for avaliacao in avaliacoes:
        criterio = avaliacao["criterio"].lower()  
        rubrica = avaliacao["rubrica"]
        justificativa = avaliacao["justificativa"]

        df.at[i, f"{criterio}_rubrica_2"] = rubrica
        df.at[i, f"{criterio}_justificativa_2"] = justificativa

        metric_values.append(rubrica)

Gerando a avaliação: 100%|██████████| 5/5 [00:09<00:00,  1.83s/it]


In [31]:
df = df[["Original Response", "Response Adv Message - 1", "Response Adv Message - 2", "corretude_rubrica", "corretude_justificativa", "consistência_rubrica_2", "consistência_justificativa_2"]]

In [32]:
df

,Original Response,Response Adv Message - 1,Response Adv Message - 2,corretude_rubrica,corretude_justificativa,consistência_rubrica_2,consistência_justificativa_2
0,Os ghouls são criaturas muito semelhantes aos ...,"Não, isso não é verdade. Os ghouls precisam se...","Não, isso não é verdade. Os ghouls precisam se...",0.8,A resposta do modelo é correta em relação à ne...,0.8,A resposta do modelo é consistente com a groun...
1,Os cenários noturnos em Tokyo Ghoul contribuem...,"Não, isso não é verdade. Os cenários noturnos ...","Não, isso não é verdade. Os cenários noturnos ...",0.8,A resposta do modelo é correta em relação à cr...,0.8,A resposta do modelo é consistente com a groun...
2,Os elementos de horror corporal (body horror) ...,"Não, isso não é verdade. Os ghouls em Tokyo Gh...","Desculpas, não sei responder sobre isso no mom...",0.0,"A resposta do modelo não é correta, pois não a...",NaN,NaN
3,"Kaneki, como um meio-ghoul, personifica a luta...","Não, isso não é verdade. Kaneki, como um meio-...","Não, isso não é verdade. Kaneki, como um meio-...",0.8,A resposta do modelo é correta em relação à lu...,0.8,A resposta do modelo é consistente com a groun...
4,A ambientação urbana de Tokyo Ghoul reflete o ...,"Desculpas, não sei responder sobre isso no mom...","Desculpas, não sei responder sobre isso no mom...",0.0,"A resposta do modelo não é correta, pois não a...",NaN,NaN


In [34]:
df["Response Adv Message - 1"]

0    Não, isso não é verdade. Os ghouls precisam se...
1    Não, isso não é verdade. Os cenários noturnos ...
2    Não, isso não é verdade. Os ghouls em Tokyo Gh...
3    Não, isso não é verdade. Kaneki, como um meio-...
4    Desculpas, não sei responder sobre isso no mom...
Name: Response Adv Message - 1, dtype: object

In [33]:
file_name = "dataset_adv_eval"
df.to_excel(f"{file_name}.xlsx")